# 🏢 Caso de Estudio: LegacyTech Solutions

## Objetivo

Demostrar la aplicación práctica del framework **EBLET** mediante un caso de estudio simulado, reproduciendo el proceso completo de diagnóstico organizacional, identificación de riesgos psicosociales y estimación del impacto económico.

## Empresa analizada: LegacyTech Solutions

**Sector:** Tecnología y desarrollo de software

**Tamaño:** 180 empleados

**Cultura declarada por la dirección:** Adhocracia (organización orientada a la innovación y la flexibilidad)

**Cultura percibida por los empleados:** Jerárquica (predominio de procesos rígidos y elevada burocracia)

**Perfil organizacional esperado:** Riesgo de boreout

## Flujo del análisis

1. Generación del conjunto de empleados de la organización.
2. Cálculo de los indicadores (KPIs) y clasificación respecto al benchmark.
3. Evaluación de la brecha entre la cultura declarada y la cultura percibida.
4. Estimación de los costes asociados a la rotación.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# ============================================================
# RUTAS DEL PROYECTO
# ============================================================

RUTA_ACTUAL = Path.cwd()

RAIZ_PROYECTO = RUTA_ACTUAL.parents[1]

RUTA_SRC = RAIZ_PROYECTO / "Python" / "src"
RUTA_DATASETS = RAIZ_PROYECTO / "Python" / "datasets"
RUTA_VISUALIZACIONES = RAIZ_PROYECTO / "visualizaciones"

sys.path.insert(0, str(RUTA_SRC))


## 1. Generación de la empresa simulada

LegacyTech Solutions se modela como una organización tecnológica de 180 empleados. La dirección declara una cultura de adhocracia, asociada a la innovación, la autonomía y la flexibilidad. Sin embargo, la experiencia cotidiana de los empleados se configura como predominantemente jerárquica, caracterizada por una mayor formalización, rigidez procedimental y dependencia de los niveles de supervisión.

Para representar un entorno con riesgo de boreout, se asignan estados latentes con niveles elevados de aburrimiento laboral, bienestar moderadamente bajo e intención de cambio elevada. Posteriormente, se introduce variabilidad individual para evitar una población completamente homogénea.

In [2]:
# ============================================================
# GENERACIÓN DE LEGACYTECH SOLUTIONS
# ============================================================

from encuesta import generar_respuestas_encuesta

N_EMPLEADOS = 180
SEMILLA = 42

np.random.seed(SEMILLA)


# ============================================================
# 1. METADATOS Y ESTADOS LATENTES
# ============================================================

df_base = pd.DataFrame({
    "empleado_id": range(1, N_EMPLEADOS + 1),
    "empresa": "LegacyTech Solutions",

    # Cultura percibida por los empleados
    "cultura": "Jerarquica",

    # Variables demográficas
    "departamento": np.random.choice(
        [
            "Desarrollo",
            "Datos",
            "Producto",
            "RRHH",
            "Ventas",
        ],
        size=N_EMPLEADOS,
        p=[0.35, 0.25, 0.20, 0.10, 0.10],
    ),

    "seniority": np.random.choice(
        [
            "Junior",
            "Mid",
            "Senior",
            "Lead",
        ],
        size=N_EMPLEADOS,
        p=[0.30, 0.40, 0.20, 0.10],
    ),

    "modalidad": np.random.choice(
        [
            "Presencial",
            "Híbrido",
            "Remoto",
        ],
        size=N_EMPLEADOS,
        p=[0.40, 0.45, 0.15],
    ),

    # Estados latentes asociados a riesgo de boreout
    "L_burnout": 2.2,
    "L_boreout": 3.8,
    "L_wellbeing": 2.6,
    "L_rotation": 3.6,
})


# ============================================================
# 2. VARIABILIDAD INDIVIDUAL
# ============================================================

columnas_latentes = [
    "L_burnout",
    "L_boreout",
    "L_wellbeing",
    "L_rotation",
]

for columna in columnas_latentes:
    df_base[columna] = (
        df_base[columna]
        + np.random.normal(
            loc=0,
            scale=0.5,
            size=N_EMPLEADOS,
        )
    ).clip(1, 5)


# ============================================================
# 3. RESPUESTAS DE LA ENCUESTA
# ============================================================

df_respuestas = generar_respuestas_encuesta(df_base)

df_legacy = pd.concat(
    [
        df_base.reset_index(drop=True),
        df_respuestas.reset_index(drop=True),
    ],
    axis=1,
)


# ============================================================
# 4. CULTURA DECLARADA Y SALARIOS
# ============================================================

df_legacy["cultura_declarada"] = "Adhocracia"

df_legacy["salario_anual"] = (
    np.random.normal(
        loc=35_000,
        scale=10_000,
        size=N_EMPLEADOS,
    )
    .clip(20_000, 60_000)
    .round()
    .astype(int)
)


# ============================================================
# 5. COMPROBACIONES
# ============================================================

columnas_preguntas = sorted(
    [
        columna
        for columna in df_legacy.columns
        if columna.startswith("q")
        and columna[1:].isdigit()
    ],
    key=lambda columna: int(columna[1:]),
)

resumen_generacion = pd.DataFrame({
    "Indicador": [
        "Número de empleados",
        "Número de preguntas",
        "Cultura declarada",
        "Cultura percibida",
        "Salario medio anual",
    ],
    "Resultado": [
        len(df_legacy),
        len(columnas_preguntas),
        df_legacy["cultura_declarada"].iloc[0],
        df_legacy["cultura"].iloc[0],
        f"{df_legacy['salario_anual'].mean():,.0f} €",
    ],
})

display(resumen_generacion)

📁 Directorio actual: C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python\notebooks
📁 Ruta real de datasets: C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python\notebooks\datasets


,Indicador,Resultado
0,Número de empleados,180
1,Número de preguntas,67
2,Cultura declarada,Adhocracia
3,Cultura percibida,Jerarquica
4,Salario medio anual,"35,485 €"


In [3]:
# ============================================================
# COMPROBACIÓN DE LAS DIMENSIONES CULTURALES CVF
# ============================================================

resumen_cultura_cvf = pd.DataFrame({
    "Dimensión cultural": [
        "Adhocracia",
        "Clan",
        "Mercado",
        "Jerárquica",
    ],
    "Ítems": [
        "q60-q61",
        "q62-q63",
        "q64-q65",
        "q66-q67",
    ],
    "Puntuación media": [
        df_legacy[["q60", "q61"]].mean().mean(),
        df_legacy[["q62", "q63"]].mean().mean(),
        df_legacy[["q64", "q65"]].mean().mean(),
        df_legacy[["q66", "q67"]].mean().mean(),
    ],
})

resumen_cultura_cvf["Puntuación media"] = (
    resumen_cultura_cvf["Puntuación media"]
    .round(2)
)

resumen_cultura_cvf = (
    resumen_cultura_cvf
    .sort_values(
        "Puntuación media",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(resumen_cultura_cvf)

,Dimensión cultural,Ítems,Puntuación media
0,Jerárquica,q66-q67,3.98
1,Mercado,q64-q65,2.02
2,Clan,q62-q63,2.00
3,Adhocracia,q60-q61,1.99


La comprobación de los ítems culturales permite verificar que la dimensión jerárquica presenta la puntuación media más elevada. Este resultado confirma que las respuestas simuladas son coherentes con la cultura organizacional percibida definida para el caso de estudio.

Esta diferencia entre una cultura declarada de adhocracia y una cultura real jerárquica constituye la brecha cultural central del caso LegacyTech.

## 2. 📊 Cálculo de KPIs y Clasificación

In [4]:
# ============================================================
# CÁLCULO DE KPIs Y PERFILES INDIVIDUALES
# ============================================================

from scores import calcular_kpis_empleado
from clasificador_individual import clasificar_individuo


# Calcular KPIs a partir de las respuestas de la encuesta
df_legacy = calcular_kpis_empleado(df_legacy)


# ============================================================
# RESUMEN DE KPIs
# ============================================================

resumen_kpis = pd.DataFrame({
    "KPI": [
        "Contexto",
        "Burnout",
        "Boreout",
        "Bienestar",
        "Intención de cambio",
    ],
    "Puntuación media": [
        df_legacy["kpi_contexto"].mean(),
        df_legacy["kpi_burnout"].mean(),
        df_legacy["kpi_boreout"].mean(),
        df_legacy["kpi_bienestar"].mean(),
        df_legacy["kpi_rotacion"].mean(),
    ],
})

resumen_kpis["Puntuación media"] = (
    resumen_kpis["Puntuación media"]
    .round(2)
)

display(resumen_kpis)

,KPI,Puntuación media
0,Contexto,2.45
1,Burnout,2.09
2,Boreout,3.80
3,Bienestar,2.56
4,Intención de cambio,3.67


In [5]:
# ============================================================
# CLASIFICACIÓN DE PERFILES INDIVIDUALES
# ============================================================

def obtener_perfil_individual(fila):

    kpis_individuales = {
        "burnout": fila["kpi_burnout"],
        "boreout": fila["kpi_boreout"],
        "bienestar": fila["kpi_bienestar"],
        "rotacion": fila["kpi_rotacion"],
        "cultura_dominante": fila["cultura_percibida"],
    }

    resultado = clasificar_individuo(
        kpis_individuales
    )

    return resultado["nombre"]


df_legacy["perfil_individual"] = (
    df_legacy.apply(
        obtener_perfil_individual,
        axis=1,
    )
)

distribucion_perfiles = (
    df_legacy["perfil_individual"]
    .value_counts()
    .rename_axis("Perfil individual")
    .reset_index(name="Número de empleados")
)

distribucion_perfiles["Porcentaje (%)"] = (
    distribucion_perfiles["Número de empleados"]
    / len(df_legacy)
    * 100
).round(1)

display(distribucion_perfiles)

,Perfil individual,Número de empleados,Porcentaje (%)
0,🔵 Aburrido Crónico,108,60.0
1,🟡 Estable Neutro,39,21.7
2,⚫ Desvinculado,27,15.0
3,🟡 Estable,5,2.8
4,🔴 Riesgo Dual,1,0.6


## 2. Cálculo de indicadores y perfiles individuales

A partir de las respuestas generadas para los 67 ítems de la encuesta, se calculan los principales indicadores del framework EBLET Enterprise: contexto organizacional, burnout, boreout, bienestar e intención de cambio laboral.

El cálculo del burnout considera sus tres dimensiones —agotamiento, cinismo e ineficacia profesional—, invirtiendo previamente los ítems de eficacia para que una puntuación elevada represente un mayor nivel de riesgo.

Posteriormente, cada empleado es asignado a un perfil individual en función de la combinación de sus indicadores.

In [6]:
# ============================================================
# EXPORTACIÓN DEL DATASET DE LEGACYTECH
# ============================================================

from pathlib import Path

columnas_exportacion = [
    "empleado_id",
    "empresa",
    "departamento",
    "seniority",
    "modalidad",
    "cultura",
    "cultura_declarada",

    *[f"q{i}" for i in range(1, 68)],

    "kpi_contexto",
    "kpi_burnout",
    "kpi_boreout",
    "kpi_bienestar",
    "kpi_rotacion",

    "perfil_individual",
]

# Conservar únicamente las columnas existentes
columnas_exportacion = [
    c
    for c in columnas_exportacion
    if c in df_legacy.columns
]

ruta_salida = (
    Path(RUTA_DATASETS)
    / "legacy_tech_ligero.csv"
)

df_legacy[columnas_exportacion].to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8",
)

resumen_exportacion = pd.DataFrame({
    "Indicador": [
        "Archivo",
        "Registros",
        "Columnas",
    ],
    "Resultado": [
        ruta_salida.name,
        len(df_legacy),
        len(columnas_exportacion),
    ],
})

display(resumen_exportacion)

,Indicador,Resultado
0,Archivo,legacy_tech_ligero.csv
1,Registros,180
2,Columnas,80


In [7]:

# ============================================================
# CARGA DEL DATASET BENCHMARK
# ============================================================

from pathlib import Path

ESCENARIOS = [
    "saludable",
    "estable",
    "riesgo_burnout",
    "riesgo_boreout",
    "critico",
]

dfs = []

for escenario in ESCENARIOS:

    ruta = (
        Path(RUTA_DATASETS)
        / escenario
        / "empleados.csv"
    )

    df = pd.read_csv(ruta)
    df["escenario"] = escenario
    dfs.append(df)

df_benchmark = pd.concat(
    dfs,
    ignore_index=True,
)

resumen_benchmark = pd.DataFrame({
    "Indicador": [
        "Empleados",
        "Escenarios",
    ],
    "Resultado": [
        len(df_benchmark),
        df_benchmark["escenario"].nunique(),
    ],
})

display(resumen_benchmark)

,Indicador,Resultado
0,Empleados,12500
1,Escenarios,5


In [8]:
# ============================================================
# POSICIONAMIENTO DE LEGACYTECH FRENTE AL BENCHMARK
# ============================================================

# Medias de LegacyTech
metricas_legacy = {
    "burnout": df_legacy["kpi_burnout"].mean(),
    "boreout": df_legacy["kpi_boreout"].mean(),
    "bienestar": df_legacy["kpi_bienestar"].mean(),
    "rotacion": df_legacy["kpi_rotacion"].mean(),
}

# Centroides de los escenarios benchmark
centroides_benchmark = (
    df_benchmark
    .groupby("escenario")[
        ["kpi_burnout", "kpi_boreout"]
    ]
    .mean()
    .reindex(ESCENARIOS)
)

# Determinar el cuadrante de LegacyTech
if (
    metricas_legacy["burnout"] < 3.5
    and metricas_legacy["boreout"] < 3.0
):
    cuadrante = "Saludable"

elif (
    metricas_legacy["burnout"] >= 3.5
    and metricas_legacy["boreout"] < 3.0
):
    cuadrante = "Riesgo de burnout"

elif (
    metricas_legacy["burnout"] < 3.5
    and metricas_legacy["boreout"] >= 3.0
):
    cuadrante = "Riesgo de boreout"

else:
    cuadrante = "Crítico"


resumen_posicionamiento = pd.DataFrame({
    "Indicador": [
        "Burnout",
        "Boreout",
        "Bienestar",
        "Intención de cambio",
        "Cuadrante",
    ],
    "Resultado": [
        round(metricas_legacy["burnout"], 2),
        round(metricas_legacy["boreout"], 2),
        round(metricas_legacy["bienestar"], 2),
        round(metricas_legacy["rotacion"], 2),
        cuadrante,
    ],
})

display(resumen_posicionamiento)

,Indicador,Resultado
0,Burnout,2.09
1,Boreout,3.8
2,Bienestar,2.56
3,Intención de cambio,3.67
4,Cuadrante,Riesgo de boreout


### 3. Posicionamiento de LegacyTech respecto al benchmark

La siguiente visualización representa la posición de LegacyTech en relación con el conjunto de empresas del benchmark según sus niveles medios de **burnout** y **boreout**. Las líneas de referencia situadas en **3,5** delimitan los umbrales de riesgo para ambos indicadores.

![Posicionamiento de LegacyTech frente al benchmark](../../visualizaciones/12_posicionamiento_legacytech_benchmark.png)

**Figura 1.** Posicionamiento de LegacyTech respecto al benchmark. Elaboración propia.

LegacyTech se sitúa en el cuadrante de **riesgo de boreout**, caracterizado por niveles elevados de boreout y niveles de burnout por debajo del umbral de riesgo. Este posicionamiento sugiere la presencia de problemas relacionados con la infraocupación, la falta de retos y la pérdida de motivación en una parte de la organización.

In [14]:
from pathlib import Path

rutas = [
    Path("../visualizaciones/legacytech_burnout_departamento_seniority.png"),
    Path("../visualizaciones/legacytech_boreout_departamento_seniority.png"),
    Path("../visualizaciones/legacytech_bienestar_departamento_seniority.png"),
]



### 4 Análisis de KPIs por departamento y seniority

La siguiente visualización muestra los valores medios de **burnout**, **boreout** y **bienestar** para cada combinación de departamento y nivel de seniority en LegacyTech. La intensidad del color representa el nivel medio alcanzado en cada indicador, facilitando la identificación de áreas y colectivos con mayor riesgo o mejor estado de bienestar.

![Burnout medio por departamento y seniority](../visualizaciones/legacytech_burnout_departamento_seniority.png?v=2)

**Figura 2.** Burnout medio por departamento y seniority. **Fuente:** Elaboración propia.

![Boreout medio por departamento y seniority](../visualizaciones/legacytech_boreout_departamento_seniority.png?v=2)

**Figura 3.** Boreout medio por departamento y seniority. **Fuente:** Elaboración propia.

![Bienestar medio por departamento y seniority](../visualizaciones/legacytech_bienestar_departamento_seniority.png?v=2)

**Figura 4.** Bienestar medio por departamento y seniority. **Fuente:** Elaboración propia.



El mapa de calor permite identificar de forma visual las diferencias existentes entre departamentos y niveles de seniority. Las celdas con valores más elevados en **burnout** y **boreout**, así como aquellas con menores niveles de **bienestar**, señalan los colectivos que pueden requerir un análisis más detallado y la priorización de medidas específicas de intervención.

## 5. 🏛️ Diagnóstico de Brecha Cultural

Mide la diferencia entre la cultura que la empresa *cree* que tiene y la que los empleados *realmente perciben*.

### Brecha entre la cultura declarada y la cultura percibida

La siguiente visualización compara la **cultura organizacional declarada** por LegacyTech con la **cultura percibida** por sus empleados, utilizando las cuatro dimensiones del modelo **Competing Values Framework (CVF)**. Esta comparación permite identificar posibles desajustes entre la identidad organizacional que la empresa pretende transmitir y la experiencia real de sus profesionales.

![Brecha entre la cultura declarada y la cultura percibida](../visualizaciones/legacytech_cultura_declarada_vs_percibida.png)

**Figura 5.** Comparación entre la cultura declarada y la cultura percibida en LegacyTech según el modelo CVF. **Fuente:** Elaboración propia.

Las diferencias entre ambos perfiles ponen de manifiesto el grado de alineación entre la cultura que la organización considera promover y la que realmente perciben sus empleados. Las mayores discrepancias identifican las dimensiones culturales sobre las que conviene focalizar futuras iniciativas de transformación y comunicación interna.

## 6. 💰 Análisis de Costes de Rotación

¿Cuánto le cuesta a LegacyTech el problema detectado?

In [12]:
from costes_rotacion import (
    kpi_a_tasa_rotacion,
    tasa_rotacion_a_categoria
)

kpi_rotacion_medio = df_legacy["kpi_rotacion"].mean()
tasa_rotacion = kpi_a_tasa_rotacion(kpi_rotacion_medio)
categoria_rotacion = tasa_rotacion_a_categoria(tasa_rotacion)

n_empleados = len(df_legacy)
n_bajas_estimadas = round(n_empleados * tasa_rotacion)

print(f"KPI medio de intención de rotación: {kpi_rotacion_medio:.2f}")
print(f"Tasa anual estimada: {tasa_rotacion:.1%}")
print(f"Categoría: {categoria_rotacion}")
print(f"Número de empleados: {n_empleados}")
print(f"Bajas anuales estimadas: {n_bajas_estimadas}")

KPI medio de intención de rotación: 3.67
Tasa anual estimada: 35.0%
Categoría: Alta
Número de empleados: 180
Bajas anuales estimadas: 63


### 6.1 Rotación anual estimada

La siguiente visualización muestra la **tasa anual estimada de rotación** de LegacyTech, calculada a partir del indicador de intención de rotación obtenido mediante el modelo EBLET. Este indicador permite estimar el porcentaje de empleados que podrían abandonar la organización en un periodo anual si no se implantan medidas de mejora.

![Rotación anual estimada en LegacyTech](../visualizaciones/legacytech_tasa_rotacion_estimada.png)

**Figura 6.** Rotación anual estimada de LegacyTech. **Fuente:** Elaboración propia.


La organización presenta una **rotación anual estimada del 35 %**, equivalente a aproximadamente **63 bajas** sobre una plantilla de **180 empleados**. Este resultado sitúa a LegacyTech en un nivel de riesgo **alto o crítico**, constituyendo la base para estimar el impacto económico asociado a la pérdida de talento en los análisis siguientes.

### 6.2 Coste anual estimado de la rotación por departamento

La siguiente visualización presenta el **coste anual estimado de la rotación** para cada departamento de LegacyTech. El cálculo se basa en el modelo económico definido en el framework EBLET, que estima el coste esperado de sustitución de cada empleado a partir de su salario, nivel de seniority y la probabilidad de abandono derivada del KPI de intención de rotación. La agregación por departamentos permite identificar las áreas donde la pérdida de talento tendría un mayor impacto económico para la organización.

![Coste anual estimado de la rotación por departamento](../visualizaciones/legacytech_coste_rotacion_departamento.png)

**Figura 7.** Coste anual estimado de la rotación por departamento. **Fuente:** Elaboración propia.

Los resultados evidencian que el impacto económico de la rotación no se distribuye de forma homogénea entre los distintos departamentos. Aquellas áreas con mayores costes concentran un mayor riesgo financiero asociado a la pérdida de talento, ya sea por el volumen de empleados, por el nivel salarial o por una mayor intención de abandono. En consecuencia, estos departamentos representan la prioridad para la implantación de medidas de retención, ya que cualquier reducción de la rotación generará un retorno económico más significativo para la organización.

### 6.3 Impacto económico potencial de una intervención

Con el objetivo de valorar la rentabilidad de implantar medidas dirigidas a mejorar el bienestar, reducir los riesgos psicosociales y fortalecer la cultura organizativa, se plantea un **escenario moderado de intervención**. En este escenario se asume una inversión equivalente al **10 % del coste anual estimado de la rotación**, con una reducción potencial del **30 % de la rotación**.

Estos valores no representan una predicción exacta, sino una **simulación económica** que permite estimar el posible retorno de las iniciativas de gestión del talento. Este tipo de análisis es habitual en la evaluación de proyectos, ya que facilita comparar el coste de una intervención con los beneficios económicos que podría generar.

![Impacto económico potencial de la intervención](../visualizaciones/legacytech_impacto_economico_estimado.png)

**Figura 8.** Simulación del impacto económico de una intervención orientada a reducir la rotación. **Fuente:** Elaboración propia.

Los resultados muestran que una inversión relativamente moderada en políticas de personas puede traducirse en un ahorro económico significativo al disminuir la rotación del personal. Además de reducir los costes asociados a los procesos de selección, incorporación y pérdida de conocimiento, la organización obtiene un retorno positivo de la inversión realizada.

Este análisis pone de manifiesto que las actuaciones sobre factores como el bienestar, el burnout, el boreout o la cultura organizativa no solo contribuyen a mejorar la experiencia de los empleados, sino que también pueden generar beneficios económicos cuantificables. En consecuencia, la gestión de personas deja de considerarse únicamente un gasto para convertirse en una inversión estratégica con impacto directo sobre los resultados de la organización.

## Conclusiones del caso LegacyTech

El análisis realizado ha permitido identificar los principales riesgos organizativos de LegacyTech mediante la integración de indicadores de bienestar, burnout, boreout, intención de rotación y cultura organizativa.

Los resultados muestran que existen diferencias significativas entre departamentos, identificándose áreas prioritarias de intervención con mayores niveles de riesgo psicosocial y mayor impacto económico potencial.

Asimismo, la simulación del coste de la rotación evidencia que las acciones dirigidas a mejorar el bienestar y la experiencia de las personas pueden generar beneficios organizativos y económicos, apoyando la toma de decisiones basada en datos.

